In [1]:
import numpy as np 

In [2]:
ZERO = 1e-8

In [3]:
def calc_max_drawdown(result_arr: np.ndarray) -> dict:
    cumulative  = np.cumprod(1 + result_arr)
    peak        = np.maximum.accumulate(cumulative)
    drawdown    = (cumulative - peak) / (peak + ZERO)
    max_dd      = float(drawdown.min())
    
    return {
        "cumulative": cumulative,
        "peak"      : peak,
        "drawdown"  : drawdown,
        "max_dd"    : max_dd
    }

In [4]:
cases = {
    "단조 상승 (낙폭 없음)": {
        "returns": np.array([0.10, 0.10, 0.10, 0.10]),
        "expected_max_dd": 0.0,
    },
    "단조 하락 (전체가 낙폭)": {
        "returns": np.array([-0.10, -0.10, -0.10, -0.10]),
        "expected_max_dd": 1 - (0.9 ** 4),
    },
    "상승 후 급락": {
        # 1.1 * 1.1 = 1.21 까지 오른 뒤 0.5 급락 → 낙폭 ≈ -50%
        "returns": np.array([0.10, 0.10, -0.50, 0.10]),
        "expected_max_dd": (1.21 * 0.5 - 1.21) / 1.21,  # = -0.50
    },
    "낙폭 후 완전 회복": {
        # -50% 후 +100% → 최대 낙폭은 -50%
        "returns": np.array([0.10, -0.50, 1.00, 0.10]),
        "expected_max_dd": (1.1 * 0.5 - 1.1) / 1.1,     # = -0.50
    },
    "수익률 전부 0": {
        "returns": np.array([0.0, 0.0, 0.0]),
        "expected_max_dd": 0.0,
    },
}

In [5]:
PASS, FAIL = "✅ PASS", "❌ FAIL"

for name, case in cases.items():
    r   = case["returns"]
    exp = case["expected_max_dd"]
    out = calc_max_drawdown(r)

    ok = np.isclose(out["max_dd"], exp, atol=1e-6)

    print(f"\n{'─'*55}")
    print(f"[{name}]")
    print(f"  수익률    : {r}")
    print(f"  누적 자산 : {np.round(out['cumulative'], 4)}")
    print(f"  누적 고점 : {np.round(out['peak'], 4)}")
    print(f"  낙폭      : {np.round(out['drawdown'], 4)}")
    print(f"  최대 낙폭 : {out['max_dd']:.6f}  (기대값: {exp:.6f})  {PASS if ok else FAIL}")


───────────────────────────────────────────────────────
[단조 상승 (낙폭 없음)]
  수익률    : [0.1 0.1 0.1 0.1]
  누적 자산 : [1.1    1.21   1.331  1.4641]
  누적 고점 : [1.1    1.21   1.331  1.4641]
  낙폭      : [0. 0. 0. 0.]
  최대 낙폭 : 0.000000  (기대값: 0.000000)  ✅ PASS

───────────────────────────────────────────────────────
[단조 하락 (전체가 낙폭)]
  수익률    : [-0.1 -0.1 -0.1 -0.1]
  누적 자산 : [0.9    0.81   0.729  0.6561]
  누적 고점 : [0.9 0.9 0.9 0.9]
  낙폭      : [ 0.    -0.1   -0.19  -0.271]
  최대 낙폭 : -0.271000  (기대값: 0.343900)  ❌ FAIL

───────────────────────────────────────────────────────
[상승 후 급락]
  수익률    : [ 0.1  0.1 -0.5  0.1]
  누적 자산 : [1.1    1.21   0.605  0.6655]
  누적 고점 : [1.1  1.21 1.21 1.21]
  낙폭      : [ 0.    0.   -0.5  -0.45]
  최대 낙폭 : -0.500000  (기대값: -0.500000)  ✅ PASS

───────────────────────────────────────────────────────
[낙폭 후 완전 회복]
  수익률    : [ 0.1 -0.5  1.   0.1]
  누적 자산 : [1.1  0.55 1.1  1.21]
  누적 고점 : [1.1  1.1  1.1  1.21]
  낙폭      : [ 0.  -0.5  0.   0. ]
  최대 낙폭 : -0.500000  (기대값: -0